<a href="https://colab.research.google.com/github/lsgrep/agents/blob/claude/agent-building-lessons-16749f/notebooks/03_reliability_math.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Reliability, and the horizon wall

**The claim you should be able to make when you finish:** *"Per-step accuracy
compounds. There are exactly two levers — raise `p_step` or lower `n_steps` —
and every technique in the field is one of them in disguise."*

Here is the conversation this lab is designed to end:

> "The model scores 95% on our task."
> "Great. So why does the agent fail half the time?"

Both statements are true. The gap between them is four lines of arithmetic, and
once you have seen it you cannot design an agent the same way again.

Twenty minutes, no API key.

In [ ]:
# Cell 1 — bootstrap. No GPU, no API key, no spend.
REPO, BRANCH = "https://github.com/lsgrep/agents.git", "claude/agent-building-lessons-16749f"

import os, subprocess, sys

if not os.path.isdir("agents"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "agents", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("agents"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'matplotlib', 'numpy'], check=True)

import agentlab
env = agentlab.notebook_setup()

## 1. Predict first

An agent takes 20 dependent steps. Each one succeeds with probability 0.95.

**Write down your guess for P(the task completes).**

Most people say something in the eighties.

In [ ]:
from agentlab import reliability as rel

print(rel.derive_horizon(p_step=0.95, n_steps=20, target=0.9))

**36%.**

And the two lines under it are the ones that reframe the problem:

- at 95% per step, a 90% task success rate buys you **two steps**;
- to hit 90% over 20 steps you need **99.47%** per step.

Nothing about this is a model-quality argument. It is an exponent.

## 2. The wall, drawn

The curve is not a gentle decline. It looks fine, looks fine, looks fine, and
then there is nothing left.

In [ ]:
print(f"{'p_step':>7} " + "".join(f"{n:>8}" for n in (5, 10, 20, 50, 100)))
print(f"{'':>7} " + "".join(f"{'steps':>8}" for _ in range(5)))
for p in (0.99, 0.98, 0.95, 0.90, 0.80):
    row = "".join(f"{rel.horizon_success(p, n):>8.1%}" for n in (5, 10, 20, 50, 100))
    print(f"{p:>7.0%} {row}")

Read the 99% row. **A 99%-per-step agent is a coin flip at 70 steps.** Read the
90% row: it is finished by step 50.

Now the same fact from the other direction — what a given horizon *demands*.

In [ ]:
print(f"{'steps':>7} {'p_step for 90%':>16} {'p_step for 99%':>16}")
for n in (5, 10, 20, 50, 100, 500):
    print(f"{n:>7} {rel.required_step_accuracy(n, 0.9):>15.3%} {rel.required_step_accuracy(n, 0.99):>15.4%}")

A 100-step task at 90% needs **99.9%** per step. A 500-step task at 99% needs
**99.998%**.

Now hold that next to a question nobody asks until lab 6: *how would you
measure 99.9%?* To distinguish 99.9% from 99.5% you need thousands of trials.
So for long-horizon work the arithmetic often says the design is out of reach
**before you build it** — which is the cheapest finding available anywhere in
this repo.

## 3. Lever one: raise `p_step` (it is the expensive one)

Raising per-step accuracy means better tools (lab 4), cleaner context (lab 7),
a better model, or more thinking. All real, all bounded, and the compounding
works against you: going from 95% to 97% is a 40% cut in per-step error and
still leaves you at 54% over 20 steps.

In [ ]:
print(f"{'p_step':>7} {'error/step':>11} {'P(20 steps)':>13} {'error cut':>11}")
base = 1 - 0.95
for p in (0.95, 0.97, 0.99, 0.995, 0.999):
    print(f"{p:>7.1%} {1 - p:>11.1%} {rel.horizon_success(p, 20):>13.1%} {1 - (1 - p) / base:>10.0%}")

## 4. Lever two: lower `n_steps` (it is usually the cheap one)

Halving the number of steps *squares* your task success rate — in the good
direction. This is why "give the agent one tool that does the whole job"
routinely beats "give the agent five composable tools and let it figure it out",
and it is the real reason behind advice that otherwise sounds like taste.

In [ ]:
print(f"{'design':<38} {'steps':>6} {'P(task)':>9}")
for label, n in [
    ("five granular tools, agent composes", 20),
    ("one tool that does the composition", 10),
    ("...and a cached first step", 8),
    ("one purpose-built tool", 4),
]:
    print(f"{label:<38} {n:>6} {rel.horizon_success(0.95, n):>9.1%}")

print("\nSame model. Same p_step. 36% to 81% by removing steps.")

## 5. Retries, and what a verifier is actually worth

"Just retry it" is the reflex. It works — `1 - (1-p)**attempts` is a powerful
formula — but it has a load-bearing precondition hiding in it: **you have to
detect the failure**.

An undetected failure is not retried. It is *committed*, and every later step is
built on top of it. So a retry loop is worth exactly what its verifier is worth,
and `p_detect` is a number you should measure by injecting known-bad steps and
counting how many your check catches.

In [ ]:
print(f"{'p_detect':>9} {'p_eff/step':>12} {'P(20 steps)':>13}")
print(f"{'none':>9} {0.90:>12.2%} {rel.horizon_success(0.90, 20):>13.1%}   (no retries at all)")
for detect in (0.5, 0.8, 0.95, 1.0):
    eff = rel.with_imperfect_verifier(0.90, attempts=3, p_detect=detect)
    print(f"{detect:>9.0%} {eff:>12.2%} {rel.horizon_success(eff, 20):>13.1%}")

A perfect verifier with three attempts takes a 90% step to 99.9% and a doomed
run to a 98% one. A verifier that catches half the failures gets you a fraction
of that.

**The retry loop is not the investment. The verifier is.**

## 6. Checkpoints: capping the blast radius

A checkpoint does not make any step more likely to succeed. It caps how much
work a failure destroys — you re-run one segment, not the run. That turns one
exponential of length `n` into `n/segment` exponentials of length `segment`.

In [ ]:
print(f"{'segment length':>15} {'P(24-step task)':>17}")
print(f"{'none (24)':>15} {rel.horizon_success(0.9, 24):>17.1%}")
for seg in (12, 8, 4, 2):
    print(f"{seg:>15} {rel.checkpoint_horizon(0.9, 24, segment=seg):>17.1%}")

print("\nThis is why 'write the plan to a file and work through it' outperforms")
print("'hold the plan in the transcript' — a checkpoint is durable, a transcript is not.")

## 7. Reporting it honestly: pass@k and pass^k

Two numbers, same agent. They answer different questions and only one of them is
the one production asks.

- **pass@k** — *does the ability exist?* Any one of `k` attempts succeeding
  counts. This is the number in the launch post.
- **pass^k** — *can I leave it alone?* **Every** one of `k` attempts must
  succeed. This is the number your on-call rotation experiences.

In [ ]:
p = 0.61
print(f"an agent that passes 61% of the time:")
print(f"  pass@1 = {p:.0%}")
print(f"  pass@5 = {1 - (1 - p) ** 5:.0%}   <- 'the ability is clearly there'")
print(f"  pass^5 = {rel.pass_pow_k(p, 5):.0%}   <- 'it works unsupervised'")
print(f"  pass^8 = {rel.pass_pow_k(p, 8):.0%}")

print(f"\n{'true p':>7} {'pass@1':>8} {'pass@8':>8} {'pass^8':>8}")
for p in (0.95, 0.9, 0.8, 0.61):
    print(f"{p:>7.0%} {p:>8.0%} {1 - (1 - p) ** 8:>8.0%} {rel.pass_pow_k(p, 8):>8.0%}")

Measured rather than assumed, from repeated runs per case — `pass_at_k` is the
unbiased HumanEval estimator, and `pass_pow_k_empirical` counts only the cases
that pass every time.

In [ ]:
trials = [
    [True] * 5,                              # rock solid
    [True, True, True, True, False],         # flaky — passes 80% of the time
    [True, True, False, False, False],
    [False] * 5,                             # broken
]
for k in (1, 3, 5):
    print(f"pass^{k} = {rel.pass_pow_k_empirical(trials, k):.0%}")
print("\nThe flaky case counts fully at k=1 and not at all at k=5. That is the point:")
print("a case that works four times in five is not a case you can leave alone.")

## 8. Putting it together

The honest sizing question for any agent design: *given the per-step accuracy I
can actually achieve, and the horizon this task actually needs, is the thing I
am proposing possible?*

In [ ]:
def viable(name, p_step, n_steps, attempts=1, p_detect=1.0, segment=None, target=0.9):
    eff = rel.with_imperfect_verifier(p_step, attempts, p_detect) if attempts > 1 else p_step
    p = (rel.checkpoint_horizon(eff, n_steps, segment) if segment
         else rel.horizon_success(eff, n_steps))
    print(f"{name:<44} {p:>7.1%}  {'ok' if p >= target else 'not viable at 90%'}")

print(f"{'design':<44} {'P(task)':>7}")
viable("40 steps, 95%, no verifier", 0.95, 40)
viable("40 steps, 95%, 3 retries, blind verifier", 0.95, 40, attempts=3, p_detect=0.4)
viable("40 steps, 95%, 3 retries, good verifier", 0.95, 40, attempts=3, p_detect=0.9)
viable("40 steps, 95%, checkpoints every 5", 0.95, 40, segment=5)
viable("12 steps, 95%, 3 retries, good verifier", 0.95, 12, attempts=3, p_detect=0.9)

Read the last two rows together. Cutting the task from 40 steps to 12 does more
than any amount of retry machinery bolted onto the 40-step version — and it is
usually cheaper to build.

## 9. The caveat that matters

Everything on this page assumes **independent** steps. Real agent failures
correlate: a wrong turn early poisons everything after it, and a growing
transcript makes later steps worse than earlier ones.

So treat all of this as the **optimistic bound**. If the arithmetic already says
the horizon is out of reach, no amount of measurement will rescue it.

**[Lab 5](05_the_doom_loop.ipynb) shows what the real curve looks like** — and
where it crosses this one.

In [ ]:
from agentlab.sim import horizon_sweep

print(f"{'steps':>6} {'this lab (bound)':>18} {'lab 5 (simulated)':>19} {'gap':>8}")
for row in horizon_sweep(lengths=(4, 12, 20, 28, 36), n_runs=200):
    print(f"{row['n_required']:>6} {row['independent_bound']:>17.1%} {row['measured']:>18.1%} {row['gap']:>+8.1%}")

Two surprises in that table, and they point in opposite directions:

- at short horizons the real agent does **better** than the bound, because a
  wrong step is survivable — it just takes another step;
- at long horizons it does **far worse**, because the failures feed each other.

The bound is wrong in both directions. The crossing point is where your agent
lives or dies, and lab 5 is about finding it.

## What you can now say

- *"95% per step over 20 steps is a 36% agent — per-step accuracy compounds."*
- *"There are two levers: raise p_step, or lower n_steps. Halving the steps
  squares the success rate."*
- *"A 100-step task at 90% needs 99.9% per step, which we couldn't even
  measure — so the design is wrong, not the model."*
- *"A retry loop is worth exactly what its verifier is worth; an undetected
  failure is committed, not retried."*
- *"pass@k is a capability claim, pass^k is a reliability claim, and production
  asks the second one."*

## Next

**[Lab 4](04_tool_surface.ipynb)** attacks `p_step` at its most common source:
the tool surface.